In [0]:
!pip install openpyxl

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.ml.feature import Bucketizer
from pyspark.storagelevel import StorageLevel
import math

# ============================================================
# Load Excel
# ============================================================

excel_path = (
    "actual_vs_pred_sorted_by_actual_and_pred_desc_P3_P12_new_fts.xlsx"   # <-- change file name
)

pdf = pd.read_excel(excel_path)

print("Excel shape:", pdf.shape)

# Convert to Spark
actual_pred_df = spark.createDataFrame(pdf)

print("Spark shape:")
print((actual_pred_df.count(), len(actual_pred_df.columns)))

display(actual_pred_df.limit(5))

# ============================================================
# Config
# ============================================================

actual_col = "actual_TOTAL_NBRX_C4"
pred_col = "predicted_TOTAL_NBRX_C4"

source_df = actual_pred_df

# ============================================================
# Keep only needed columns and cache
# ============================================================

base_df = (
    source_df
    .select(
        F.col(actual_col).cast("double").alias(actual_col),
        F.col(pred_col).cast("double").alias(pred_col)
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = base_df.count()

df_nonzero = (
    base_df
    .filter(F.col(actual_col) > 0)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = df_nonzero.count()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.storagelevel import StorageLevel

# ============================================================
# Config
# ============================================================

provider_col = "PROVIDER_ID"
actual_col = "actual_TOTAL_NBRX_C4"
pred_col = "predicted_TOTAL_NBRX_C4"

# Source Spark dataframe
source_df = actual_pred_df

# ============================================================
# Keep only needed columns and cache
# ============================================================

base_df = (
    source_df
    .select(
        F.col(provider_col).alias(provider_col),
        F.col(actual_col).cast("double").alias(actual_col),
        F.col(pred_col).cast("double").alias(pred_col)
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = base_df.count()

nonzero_df = (
    base_df
    .filter(F.col(actual_col) > 0)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = nonzero_df.count()

print("Base shape :", (base_df.count(), len(base_df.columns)))
print("Nonzero shape :", (nonzero_df.count(), len(nonzero_df.columns)))

# ============================================================
# Create top 50, middle 50, bottom 50
# Middle 50 is taken from nonzero rows around the median rank
# ============================================================

# Bottom 50 nonzero: lowest actual values
bottom_50_df = (
    nonzero_df
    .orderBy(
        F.col(actual_col).asc(),
        F.col(pred_col).asc(),
        F.col(provider_col).asc()
    )
    .limit(50)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = bottom_50_df.count()

# Top 50: highest actual values
top_50_df = (
    nonzero_df
    .orderBy(
        F.col(actual_col).desc(),
        F.col(pred_col).desc(),
        F.col(provider_col).asc()
    )
    .limit(50)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = top_50_df.count()

# Middle 50: centered around median rank of nonzero rows
w_rank = Window.orderBy(
    F.col(actual_col).asc(),
    F.col(pred_col).asc(),
    F.col(provider_col).asc()
)

nonzero_ranked_df = (
    nonzero_df
    .withColumn("rn", F.row_number().over(w_rank))
    .withColumn("total_nonzero_rows", F.count("*").over(Window.partitionBy()))
)

mid_start_expr = F.floor((F.col("total_nonzero_rows") - 50) / 2) + 1
mid_end_expr = mid_start_expr + 49

middle_50_df = (
    nonzero_ranked_df
    .filter(
        (F.col("rn") >= mid_start_expr) &
        (F.col("rn") <= mid_end_expr)
    )
    .select(provider_col, actual_col, pred_col)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = middle_50_df.count()

print("Top 50 shape   :", (top_50_df.count(), len(top_50_df.columns)))
print("Middle 50 shape:", (middle_50_df.count(), len(middle_50_df.columns)))
print("Bottom 50 shape:", (bottom_50_df.count(), len(bottom_50_df.columns)))

# ============================================================
# Helper: exact bucket summaries using ntile
# ============================================================

def summarize_ntile_buckets(df, analysis_name, n_buckets):
    """
    Split the dataframe into exactly n_buckets using ntile()
    based on actual_col.
    """

    w = Window.orderBy(
        F.col(actual_col).asc(),
        F.col(pred_col).asc(),
        F.col(provider_col).asc()
    )

    bucketed_df = df.withColumn(
        "bucket_id",
        F.ntile(n_buckets).over(w)
    )

    summary_df = (
        bucketed_df
        .groupBy("bucket_id")
        .agg(
            F.count("*").alias("row_count"),
            F.countDistinct(provider_col).alias("distinct_provider_count"),
            F.sum(actual_col).alias("sum_actual_sales"),
            F.sum(pred_col).alias("sum_predicted_sales"),
            F.min(actual_col).alias("min_actual_sales"),
            F.max(actual_col).alias("max_actual_sales"),
            F.avg(actual_col).alias("avg_actual_sales"),
            F.avg(pred_col).alias("avg_predicted_sales")
        )
        .orderBy("bucket_id")
    )

    label_rows = [(i, f"bucket {i}") for i in range(1, n_buckets + 1)]
    label_df = spark.createDataFrame(label_rows, ["bucket_id", "bucket_label"])

    final_df = (
        summary_df
        .join(label_df, on="bucket_id", how="left")
        .select(
            F.lit(analysis_name).alias("analysis_name"),
            "bucket_id",
            "bucket_label",
            "row_count",
            "distinct_provider_count",
            "sum_actual_sales",
            "sum_predicted_sales",
            "min_actual_sales",
            "max_actual_sales",
            "avg_actual_sales",
            "avg_predicted_sales"
        )
        .orderBy("bucket_id")
    )

    return final_df

# ============================================================
# Bucket summaries for top / middle / bottom 50
# ============================================================

top_50_summary_3 = summarize_ntile_buckets(
    df=top_50_df,
    analysis_name="top_50_hcps_3_buckets",
    n_buckets=3
)

top_50_summary_10 = summarize_ntile_buckets(
    df=top_50_df,
    analysis_name="top_50_hcps_10_buckets",
    n_buckets=10
)

middle_50_summary_3 = summarize_ntile_buckets(
    df=middle_50_df,
    analysis_name="middle_50_hcps_3_buckets",
    n_buckets=3
)

middle_50_summary_10 = summarize_ntile_buckets(
    df=middle_50_df,
    analysis_name="middle_50_hcps_10_buckets",
    n_buckets=10
)

bottom_50_summary_3 = summarize_ntile_buckets(
    df=bottom_50_df,
    analysis_name="bottom_50_hcps_nonzero_3_buckets",
    n_buckets=3
)

bottom_50_summary_10 = summarize_ntile_buckets(
    df=bottom_50_df,
    analysis_name="bottom_50_hcps_nonzero_10_buckets",
    n_buckets=10
)

# ============================================================
# Show results
# ============================================================

print("=== TOP 50 | 3 BUCKETS ===")
display(top_50_summary_3)

print("=== TOP 50 | 10 BUCKETS ===")
display(top_50_summary_10)

print("=== MIDDLE 50 | 3 BUCKETS ===")
display(middle_50_summary_3)

print("=== MIDDLE 50 | 10 BUCKETS ===")
display(middle_50_summary_10)

print("=== BOTTOM 50 (NONZERO) | 3 BUCKETS ===")
display(bottom_50_summary_3)

print("=== BOTTOM 50 (NONZERO) | 10 BUCKETS ===")
display(bottom_50_summary_10)

# ============================================================
# Optional combined output
# ============================================================

all_top_middle_bottom_summaries = (
    top_50_summary_3
    .unionByName(top_50_summary_10)
    .unionByName(middle_50_summary_3)
    .unionByName(middle_50_summary_10)
    .unionByName(bottom_50_summary_3)
    .unionByName(bottom_50_summary_10)
)

print("=== COMBINED OUTPUT ===")
display(all_top_middle_bottom_summaries)